# WF3 - Complete workflow: ParametricSimulation (aPMV setpoints)

WF3 runs a parametric study of the aPMV setpoints: thermostats that keep the adaptive PMV of each zone between −s and +s. The aPMV setpoint s and the adaptive coefficient of the cooling season λcool are varied over a grid, and the annual HVAC electricity, the annual mean operative temperature and the hours with the aPMV within its setpoints are compared.

The notebook follows Example 3 of the accim paper (parametric analysis of the aPMV setpoints), with the sample building and weather files distributed with accim instead of the office building of the paper. The results are therefore similar in nature, not identical. As in the paper, the evaluations are not screened against an adaptive comfort band: the aPMV setpoints do not control the temperature, so an adaptive band would compare two different comfort criteria.

To keep the run light, only annual results are kept: the simulation folders are deleted once the results have been read.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, paths, a lighter simulation setup (2 timesteps per hour, full-year run period) and the aPMV setpoints.
- Configuration: outputs, the aPMV parameters and the full grid of combinations.
- Execution and review: annual simulations, results in kWh/m2 and the response of the outputs to s and λcool.

Main classes, methods, and functions used in this notebook:
- `ParametricSimulation`
- `ef.get_building`
- `reduce_runtime`
- `add_vrf_system`
- `apply_apmv_setpoints`
- `set_pmv_input_parameters`
- `parametric.discover_available_outputs`
- `parametric.select_outputs`
- `parametric.apply_outputs_preflight`
- `parametric.set_output_readers`
- `parametric.set_parameters`
- `parametric.sampling_full_set`
- `ParametricSimulation.run_parametric_simulation`
- `parametric.set_building_floor_area`
- `parametric.normalize_outputs`
- `parametric.plot_parametric_lines`
- `parametric.plot_parametric_heatmap`
- `parametric.plot_parametric_scatter`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## WF3 aPMV parametric: import workflow dependencies

We start by loading every dependency required by the full workflow, so the remaining sections can run without hidden imports.

In this complete workflow notebook, imports are intentionally grouped in one place. That makes reruns in a fresh kernel predictable and easier to debug when versions change.

The cell also defines `show_plot_output`, a small helper that displays the figures and image files returned by the plotting methods.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import accim
from besos import eppy_funcs as ef
from accim.parametric_and_optimisation.main import ParametricSimulation
from accim.parametric_and_optimisation.objectives import average_results, sum_results
from accim.sim.apmv_setpoints import add_vrf_system, apply_apmv_setpoints, set_pmv_input_parameters
from accim.utils import reduce_runtime

# (Run once per notebook)
%matplotlib inline
# If you prefer interaction (zoom/pan), use: %matplotlib widget
from IPython.display import Image, display
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 160)


def show_plot_output(plot_output, image_paths=None):
    # Optional deterministic image display for methods that return tabular data.
    if image_paths is not None:
        if isinstance(image_paths, (str, Path)):
            image_paths = [image_paths]
        for plot_path in image_paths:
            display(Image(filename=str(plot_path)))

    if isinstance(plot_output, dict):
        for plot_path in plot_output.values():
            display(Image(filename=str(plot_path)))
        return
    if isinstance(plot_output, (str, Path)):
        display(Image(filename=str(plot_output)))
        return
    if hasattr(plot_output, "fig"):
        display(plot_output.fig)
        plt.show()
        return
    if hasattr(plot_output, "savefig"):
        display(plot_output)
        plt.show()
        return
    if isinstance(plot_output, pd.DataFrame):
        display(plot_output)
        return
    raise TypeError(f"Unsupported plot output type: {type(plot_output)}")


## WF3 aPMV parametric: set project paths, weather file and parallel processes

Here we declare the project assets (IDF, EPW, and results folder) used by the workflow.

The IDF and EPW files are the sample files in this folder. If accim was installed with `pip install "accim[samples]"`, copy this folder to a working directory before running the notebook, so that the installed files are not modified; the results are written to `RESULTS_DIRNAME` inside it. Only the Seville weather file is used, because the cooling season of the aPMV setpoints is given by northern-hemisphere dates (see the aPMV setup below).

Key settings in this cell and why they matter:
- `PROCESSES`: simulations run in parallel. Half of the logical CPUs is a safe default; reduce it if the computer becomes unresponsive.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_list`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.
- `PROCESSES`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm that the IDF and EPW files are found.

In [ ]:
PROJECT_ROOT = Path.cwd()

idf_list = [
    "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf",
]
epw_list = ["Seville.epw"]
RESULTS_DIRNAME = "wf3_parametric_apmv_results"

PROCESSES = max(1, (os.cpu_count() or 2) // 2)

print("IDF files found:", [(PROJECT_ROOT / idf).exists() for idf in idf_list])
print("EPW files found:", [(PROJECT_ROOT / epw).exists() for epw in epw_list])
print("Parallel processes:", PROCESSES)


## WF3 aPMV parametric: load the baseline IDF model and make the simulations lighter

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution. The changes below only affect the model in memory: the IDF file is not modified.

The simulations are made lighter while keeping annual results:
- `reduce_runtime(...)` sets 2 timesteps per hour (the model has 6), minimal shadowing and a lower limit of figures in the shadow overlap calculations. Its run period arguments keep the full year: by default, `reduce_runtime` shortens the run period to 1 January.
- `Run_Simulation_for_Sizing_Periods = "No"`: the design days are still used to size the HVAC system, but they are not simulated as additional environments. Otherwise, the annual results read after each simulation would also include the 7 design days of the sample model (8,928 hours instead of 8,760), and the HVAC electricity would be about 3 % higher.

Fewer timesteps make each simulation faster, with small differences in the annual energy compared with the 6 timesteps of the model. For final results, remove the `reduce_runtime(...)` call, but keep the sizing-period setting.

Variables created in this cell:
- `building`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm that the printed messages report 2 timesteps per hour and MinimalShadowing.

In [ ]:
building = ef.get_building(idf_list[0])

reduce_runtime(
    idf_object=building,
    minimal_shadowing=True,
    timesteps=2,
    runperiod_begin_month=1,
    runperiod_begin_day_of_month=1,
    runperiod_end_month=12,
    runperiod_end_day_of_month=31,
)
# Keep the design days out of the annual results.
building.idfobjects["SimulationControl"][0].Run_Simulation_for_Sizing_Periods = "No"


## WF3 aPMV parametric: add the VRF system and the aPMV setpoints

The aPMV (adaptive Predicted Mean Vote) corrects the PMV for the adaptation of the occupants: aPMV = PMV / (1 + λ PMV), with an adaptive coefficient λ for the cooling season and another for the heating season. The aPMV setpoints keep the aPMV of each zone between −s and +s, so the thermostat works on thermal sensation instead of on temperature.

This cell adds a VRF system with a thermal comfort thermostat to the model and the EMS programs of the aPMV setpoints, with the settings of the examples of the paper:
- heating-season adaptive coefficient λheat = −0.293;
- cooling season from 1 April to 1 October; the rest of the year is the heating season;
- the default tolerances of accim: the heating setpoint is raised by 0.1 and the cooling setpoint lowered by 0.1.

The cooling coefficient and the setpoints given here are replaced by the parameters of the study. `add_vrf_system` already adds a thermal comfort thermostat to each zone, and the model already requests some of the hourly outputs, so the warnings of `apply_apmv_setpoints` about a thermostat or an output variable that already exists are expected: it sets up those thermostats for the aPMV setpoints and skips the repeated outputs.

The PMV also depends on the inputs of the People objects. The sample model uses a constant schedule of 1 for the work efficiency, the clothing insulation and the air velocity: with a work efficiency of 1, all the metabolic heat becomes work, so the occupants would always feel cold and the thermostat would overheat the zones (to an annual mean of about 38 °C). These inputs do not affect temperature setpoints, but they do affect aPMV setpoints, so `set_pmv_input_parameters` sets a work efficiency of 0 and an air velocity of 0.1 m/s, and a seasonal schedule sets the clothing insulation: 1 clo from October to March and 0.5 clo from April to September, the cooling season. (The dynamic clothing model of EnergyPlus cannot be used here: the thermal comfort thermostats need a clothing schedule.)

The cooling season is given by dates, which only describe the seasons of the northern hemisphere. For this reason, this notebook uses only the Seville weather file (the paper also used Seville). For a location in the southern hemisphere, such as Sydney, the dates must be swapped (cooling season from 1 October to 1 April).

Key arguments used in this cell and why they matter:
- `outputs_freq`: frequency of the aPMV outputs added by accim.
- `adap_coeff_heating`: λheat.
- `pmv_cooling_sp`, `pmv_heating_sp`: initial aPMV setpoints, replaced by the parameters.
- `cooling_season_start`, `cooling_season_end`: dates of the cooling season (dd/mm).
- `air_velocity`, `work_efficiency`: PMV inputs of the occupants (m/s and fraction of the metabolic rate).
- `Clothing_Insulation_Schedule_Name`: the seasonal clothing schedule, in clo.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
add_vrf_system(building, verbose_mode=False)
apply_apmv_setpoints(
    building,
    outputs_freq=["hourly"],
    adap_coeff_heating=-0.293,
    pmv_cooling_sp=0.5,
    pmv_heating_sp=-0.5,
    cooling_season_start="01/04",
    cooling_season_end="01/10",
    verbose_mode=False,
)
set_pmv_input_parameters(building, air_velocity=0.1, work_efficiency=0.0, verbose_mode=False)
building.newidfobject(
    "Schedule:Compact",
    Name="Clothing insulation",
    Schedule_Type_Limits_Name="Any Number",
    Field_1="Through: 31 Mar", Field_2="For: AllDays", Field_3="Until: 24:00, 1.0",
    Field_4="Through: 30 Sep", Field_5="For: AllDays", Field_6="Until: 24:00, 0.5",
    Field_7="Through: 31 Dec", Field_8="For: AllDays", Field_9="Until: 24:00, 1.0",
)
for people in building.idfobjects["People"]:
    people.Clothing_Insulation_Schedule_Name = "Clothing insulation"


## WF3 aPMV parametric: initialize parametric simulation object

At this point we instantiate the parametric engine that will coordinate outputs, parameters, and run plans.

From here onward, most workflow actions are methods on this object, which keeps the execution trace coherent. The aPMV setpoints were already applied in the previous cell, so `bypass_addAccis=True` keeps the object from applying them again.

Key arguments used in this cell and why they matter:
- `buildings`: loaded IDF model used in this workflow.
- `parameters_type`: `"apmv setpoints"`.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs; the annual results are calculated from hourly values.
- `bypass_addAccis`: skips the automatic setup of the aPMV setpoints.

Variables created in this cell:
- `parametric`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `parametric`.

In [ ]:
parametric = ParametricSimulation(
    buildings=[building],
    parameters_type="apmv setpoints",
    epws=epw_list,
    output_freqs=["hourly"],
    bypass_addAccis=True,
)


## WF3 aPMV parametric: discover available outputs and select the minimal set

Before locking outputs, we scan what this model setup can actually report, and keep only the outputs the workflow uses:
- meters `Electricity:HVAC`, `Heating:Electricity` and `Cooling:Electricity`;
- `Zone Operative Temperature` of each zone;
- `Comfortable Hours_...`: an EMS output of accim that is 1 when the aPMV of the occupants of a zone is within its setpoints, so its annual sum gives the hours within the setpoints.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.
- `meters`: energy meters to include as outputs.
- `variables`: output variables to include, by name or by substring (`match="contains"`).
- `on_missing`: `"raise"` stops here if a requested output is not available.

Variables created in this cell:
- `meter_names`: reused in subsequent workflow blocks.
- `available_outputs`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.
- `df_vars_sel`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that three meters, the operative temperature of every zone and the comfortable hours of every zone are selected.

In [ ]:
meter_names = ["Electricity:HVAC", "Heating:Electricity", "Cooling:Electricity"]
available_outputs = parametric.discover_available_outputs(
    reduce_sim_time=True,
    prefer="testsimeplus",
)
print("Discovery source:", available_outputs["meta"].get("source"))

df_meters_sel = parametric.select_outputs(
    meters=meter_names,
    match="case_insensitive",
    on_missing="raise",
)["meters"]
df_vars_sel = parametric.select_outputs(
    variables=["Zone Operative Temperature", "Comfortable Hours_"],
    match="contains",
    on_missing="raise",
)["variables"]
print(df_meters_sel[["key_name", "frequency"]].to_string(index=False))
print(df_vars_sel[["key_value", "variable_name", "frequency"]].to_string(index=False))


## WF3 aPMV parametric: run output preflight checks before execution

This block applies preflight checks to the selected outputs before execution.

Preflight is the final schema gate; if this step is clean, output-related runtime errors are much less likely. With `clean_mode="all"`, the output objects of the IDF (the output variables of the sample model and those added by the aPMV setup, the SQLite file and the summary tables) are removed before the selection is added.

Key arguments used in this cell and why they matter:
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `on_missing`: behavior when a requested output is missing.

Variables created in this cell:
- `preflight_report`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Verify that the verification reports no missing or extra outputs.

In [ ]:
preflight_report = parametric.apply_outputs_preflight(
    df_meters_sel=df_meters_sel,
    df_vars_sel=df_vars_sel,
    clean_mode="all",
    validate_before_apply=False,
    validate_after_apply=True,
    on_missing="raise",
)
print(preflight_report["verification"])


## WF3 aPMV parametric: define how the annual results are read

Now we register the readers that turn the hourly outputs of each simulation into the columns of the results table: the meters and the comfortable hours are summed over the year (`sum_results`), and the operative temperature of each zone is averaged (`average_results`).

Key arguments used in this cell and why they matter:
- `df_output_meter`: meters read after each simulation, with the aggregation function in the `func` column.
- `df_output_variable`: variables read after each simulation, with `func` and the column name in `name`.

Variables created in this cell:
- `readers_meters`: reused in subsequent workflow blocks.
- `readers_variables`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm that there is one reader per meter and two per zone.

In [ ]:
readers_meters = df_meters_sel.copy()
readers_meters["func"] = [sum_results] * len(readers_meters)

readers_variables = df_vars_sel.copy()
is_temperature = readers_variables["variable_name"] == "Zone Operative Temperature"
readers_variables["func"] = [average_results if t else sum_results for t in is_temperature]
readers_variables["name"] = [
    "Annual mean operative temperature " + key if t else name.replace("Comfortable Hours_", "Hours within aPMV setpoints ")
    for t, key, name in zip(is_temperature, readers_variables["key_value"], readers_variables["variable_name"])
]

parametric.set_output_readers(
    df_output_meter=readers_meters,
    df_output_variable=readers_variables,
)
print(readers_meters["key_name"].tolist() + readers_variables["name"].tolist())


## WF3 aPMV parametric: define the aPMV parameters

This section defines the grid of the study, as in Example 3 of the paper.

Key arguments used in this cell and why they matter:
- `accis_params_dict`: aPMV parameters and their options:
  - `Adaptive cooling coefficient`: λcool, 0 (PMV setpoints, no adaptation), 0.1, 0.3 and 0.5;
  - `PMV setpoint`: s, the aPMV setpoints are −s (heating) and +s (cooling); 0.2, 0.5 and 0.7.

Variables created in this cell:
- `apmv_parameters`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm that the parameters are accepted without errors.

In [ ]:
apmv_parameters = {
    "Adaptive cooling coefficient": [0.0, 0.1, 0.3, 0.5],
    "PMV setpoint": [0.2, 0.5, 0.7],
}
parametric.set_parameters(accis_params_dict=apmv_parameters)


## WF3 aPMV parametric: configure problem definition

This section registers the parameters and the outputs as the inputs and outputs of the problem.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
parametric.set_problem()


## WF3 aPMV parametric: generate the full grid of combinations

`sampling_full_set` combines every option of every parameter with every EPW: 4 x 3 = 12 combinations.

Variables created in this cell:
- `plan_df`: reused in subsequent workflow blocks.
- `expected_runs`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the number of combinations.

In [ ]:
parametric.sampling_full_set()
plan_df = parametric.parameters_values_df

expected_runs = len(plan_df)
print("Expected simulation runs:", expected_runs)
plan_df.head()


## WF3 aPMV parametric: execute the annual runs

This is the execution checkpoint: we run the full grid and collect the annual results.

With the lighter settings, one annual simulation of the sample model takes about 25-35 s when several run in parallel: the 12 simulations take about 1-2 minutes with 6 processes. With `keep_dirs=False`, each simulation runs in a temporary folder that is deleted once its annual results have been read.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `df`: case plan dataframe executed in this run.
- `processes`: number of parallel processes for simulation runs.
- `keep_input`: whether generated input files are preserved.
- `keep_dirs`: whether per-run directories are preserved.

Variables created in this cell:
- `outputs`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare returned row count with the expected number of plan combinations.

In [ ]:
outputs = parametric.run_parametric_simulation(
    out_dir=RESULTS_DIRNAME,
    df=plan_df,
    processes=PROCESSES,
    keep_input=True,
    keep_dirs=False,
)
print("Rows:", len(outputs), "| expected:", expected_runs)


### Reading the result

Review the row count and confirm it matches the expected combinations from your plan and EPW set.

## WF3 aPMV parametric: normalize the results and collect the columns used in the plots

In this stage we convert the annual energy to kWh per m2 of floor area and add two columns: the annual mean operative temperature of the two zones and the share of the hours of the year with the aPMV within its setpoints (mean of the two zones).

Key arguments used in this cell and why they matter:
- `mode`: `"all"` uses the floor area of every zone of the IDF.
- `df_types`: dataset groups that should be normalized.

Variables created in this cell:
- `results`: the results table (`parametric.outputs_param_simulation`).
- `main_energy_col`, `heating_energy_col`, `cooling_energy_col`: energy columns.
- `temperature_col`, `comfort_col`: the two added columns.
- `plot_out_dir`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the floor area and that the share of hours within the setpoints lies between 0 and 1.

In [ ]:
parametric.set_building_floor_area(mode="all")
parametric.normalize_outputs(df_types=["parametric"])

results = parametric.outputs_param_simulation
energy_cols = [c for c in results.columns if any(meter in c for meter in meter_names)]
main_energy_col = next(c for c in energy_cols if "Electricity:HVAC" in c)
heating_energy_col = next(c for c in energy_cols if "Heating:Electricity" in c)
cooling_energy_col = next(c for c in energy_cols if "Cooling:Electricity" in c)

temperature_col = "Annual mean operative temperature"
results[temperature_col] = results[[c for c in results.columns if c.startswith(temperature_col + " ")]].mean(axis=1)
comfort_col = "Share of hours within aPMV setpoints"
results[comfort_col] = results[[c for c in results.columns if c.startswith("Hours within aPMV setpoints ")]].mean(axis=1) / 8760

plot_out_dir = Path(RESULTS_DIRNAME) / "plots"
plot_out_dir.mkdir(parents=True, exist_ok=True)
results[list(apmv_parameters) + [main_energy_col, temperature_col, comfort_col]].round(3)


## WF3 aPMV parametric: response of the outputs to s and λcool

As in Example 3 of the paper, one line per value of λcool shows how each output changes with the aPMV setpoint s. With λcool = 0, the aPMV is the PMV; larger values of λcool widen the range of temperatures accepted in the cooling season.

Key arguments used in this cell and why they matter:
- `x`: the aPMV setpoint s.
- `y_vars`: outputs drawn, one figure per output.
- `hue`, `style`: λcool.
- `col`: one panel per weather file.
- `estimator`, `errorbar`: `None` draws the simulated values, without averaging.

Variables created in this cell:
- `fig_lines`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare how the HVAC electricity and the hours within the setpoints change with s for each λcool.

In [ ]:
fig_lines = parametric.plot_parametric_lines(
    x="PMV setpoint",
    y_vars=[main_energy_col, heating_energy_col, cooling_energy_col, temperature_col, comfort_col],
    df_source="parametric",
    hue="Adaptive cooling coefficient",
    style="Adaptive cooling coefficient",
    col="epw",
    estimator=None,
    errorbar=None,
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
)
show_plot_output(fig_lines)


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF3 aPMV parametric: heatmap of the HVAC electricity

The heatmap gives the annual HVAC electricity of each combination of λcool and s.

Key arguments used in this cell and why they matter:
- `x`, `y`, `z`: λcool, s and the HVAC electricity.
- `col`: one panel per weather file.
- `annot`: writes the value of each cell.

Variables created in this cell:
- `fig_heatmap`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Look for the combinations with the lowest HVAC electricity.

In [ ]:
fig_heatmap = parametric.plot_parametric_heatmap(
    x="Adaptive cooling coefficient",
    y="PMV setpoint",
    z=main_energy_col,
    df_source="parametric",
    col="epw",
    annot=True,
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
)
show_plot_output(fig_heatmap)


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF3 aPMV parametric: HVAC electricity against the hours within the setpoints

Finally, each combination is drawn with its HVAC electricity and its share of hours with the aPMV within the setpoints, coloured by λcool and with one marker per value of s. Combinations that save energy by widening the setpoints may keep the aPMV within them for fewer hours.

Key arguments used in this cell and why they matter:
- `x`, `y`: HVAC electricity and share of hours within the setpoints.
- `hue`, `style`: λcool and s.

Variables created in this cell:
- `fig_scatter`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check whether the combinations with the lowest HVAC electricity also keep a high share of hours within the setpoints.

In [ ]:
fig_scatter = parametric.plot_parametric_scatter(
    x=main_energy_col,
    y=comfort_col,
    df_source="parametric",
    hue="Adaptive cooling coefficient",
    style="PMV setpoint",
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
)
show_plot_output(fig_scatter)


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.